# Information Theory Mastery Lab

Analyze one discrete latent-variable system as entropy, mutual information, variational inference, and code length. All primary quantities are computed directly from probability tables.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)

def entropy(p):
    p = np.asarray(p, dtype=float)
    mask = p > 0
    return -np.sum(p[mask] * np.log(p[mask]))

def kl(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float)
    mask = p > 0
    if np.any(q[mask] == 0): return np.inf
    return np.sum(p[mask] * np.log(p[mask] / q[mask]))

def mutual_information(joint):
    px = joint.sum(axis=1, keepdims=True)
    py = joint.sum(axis=0, keepdims=True)
    return kl(joint.ravel(), (px @ py).ravel())

def softmax(v):
    e = np.exp(v - np.max(v))
    return e / e.sum()


## 1. Joint model and information quantities

Rows of the likelihood are p(x|z). Logs are natural, so results are in nats unless divided by log(2).


In [ ]:
p_z = np.array([0.5, 0.3, 0.2])
p_x_given_z = np.array([
    [0.70, 0.20, 0.08, 0.02],
    [0.10, 0.60, 0.20, 0.10],
    [0.05, 0.15, 0.30, 0.50],
])
assert np.allclose(p_z.sum(), 1)
assert np.allclose(p_x_given_z.sum(axis=1), 1)

joint_zx = p_z[:, None] * p_x_given_z
p_x = joint_zx.sum(axis=0)
H_z = entropy(p_z)
H_x = entropy(p_x)
H_x_given_z = sum(p_z[z] * entropy(p_x_given_z[z]) for z in range(len(p_z)))
I_zx_entropy = H_x - H_x_given_z
I_zx_kl = mutual_information(joint_zx)

assert np.allclose(joint_zx.sum(), 1)
assert np.allclose(I_zx_entropy, I_zx_kl)
print("H(Z), H(X), H(X|Z), I(Z;X) in bits:")
print(np.array([H_z, H_x, H_x_given_z, I_zx_kl]) / np.log(2))


## 2. Data processing through a second noisy channel

The Markov chain is Z→X→Y. Exact mutual information cannot increase after the second channel.


In [ ]:
p_y_given_x = np.array([
    [0.80, 0.15, 0.05],
    [0.15, 0.70, 0.15],
    [0.10, 0.25, 0.65],
    [0.05, 0.20, 0.75],
])
assert np.allclose(p_y_given_x.sum(axis=1), 1)
joint_zy = joint_zx @ p_y_given_x
I_zy = mutual_information(joint_zy)
assert I_zy <= I_zx_kl + 1e-12
print("I(Z;X), I(Z;Y) bits:", np.array([I_zx_kl, I_zy]) / np.log(2))


## 3. Exact evidence and posterior

Bayes normalization is tractable here, giving a target against which variational inference can be checked.


In [ ]:
posterior = joint_zx / p_x[None, :]
assert np.allclose(posterior.sum(axis=0), 1)
print("p(x):", p_x)
print("p(z|x), columns are observations:\n", posterior)


## 4. Optimize a categorical variational posterior

For each observation, maximize Eq[log p(x,z)-log q]. The softmax-logit gradient is implemented directly.


In [ ]:
def optimize_q(log_joint_column, steps=1000, lr=0.2):
    logits = np.zeros_like(log_joint_column)
    history = []
    log_evidence = np.log(np.exp(log_joint_column).sum())
    for _ in range(steps):
        q = softmax(logits)
        elbo = np.sum(q * (log_joint_column - np.log(q)))
        grad_q = log_joint_column - np.log(q) - 1.0
        grad_logits = q * (grad_q - np.dot(q, grad_q))
        logits += lr * grad_logits
        history.append(log_evidence - elbo)
    q = softmax(logits)
    elbo = np.sum(q * (log_joint_column - np.log(q)))
    return q, elbo, np.array(history)

q_columns, elbos, histories = [], [], []
for x in range(len(p_x)):
    q, elbo, hist = optimize_q(np.log(joint_zx[:, x]))
    q_columns.append(q); elbos.append(elbo); histories.append(hist)
q_fit = np.array(q_columns).T
elbos = np.array(elbos)
histories = np.array(histories)

assert np.allclose(q_fit, posterior, atol=2e-4)
assert np.all(elbos <= np.log(p_x) + 1e-10)
assert np.allclose(np.log(p_x) - elbos, [kl(q_fit[:,x], posterior[:,x]) for x in range(len(p_x))], atol=1e-8)
print("maximum posterior error:", np.max(np.abs(q_fit-posterior)))


In [ ]:
for x in range(len(p_x)):
    plt.plot(histories[x], label=f"x={x}")
plt.yscale("log"); plt.xlabel("gradient step"); plt.ylabel("evidence - ELBO"); plt.legend();


## 5. Bits-back cost equals negative ELBO

Use the optimized q for each observation and verify the coding identity in the same log base.


In [ ]:
for x in range(len(p_x)):
    q = q_fit[:, x]
    reconstruction_cost = -np.sum(q * np.log(p_x_given_z[:, x]))
    latent_cost = kl(q, p_z)
    net_cost = reconstruction_cost + latent_cost
    assert np.allclose(net_cost, -elbos[x], atol=1e-10)
print("negative ELBO in bits:", -elbos / np.log(2))


## 6. A collapsed latent model

If every latent state uses the same likelihood, X contains no information about Z and the posterior equals the prior.


In [ ]:
shared_likelihood = np.tile(p_x[None, :], (len(p_z), 1))
collapsed_joint = p_z[:, None] * shared_likelihood
collapsed_posterior = collapsed_joint / collapsed_joint.sum(axis=0, keepdims=True)
assert np.allclose(collapsed_posterior, p_z[:, None])
assert np.allclose(mutual_information(collapsed_joint), 0.0, atol=1e-12)
print("collapsed I(Z;X):", mutual_information(collapsed_joint))


## Exercises

1. Verify I(Z;X) using H(Z)-H(Z|X) as a third calculation.
2. Restrict q so it must be identical for every x. Measure the approximation gap.
3. Change the second channel and plot I(Z;Y) against its noise level.
4. Compare nats and bits; identify which rankings and equalities remain unchanged.
5. Build three model variants and rank them with an explicit two-part MDL code.
6. Explain the initial-bit-reservoir requirement in practical bits-back coding.

Carry the completed notebook into the Compression and Latent-Code Audit capstone.
